# Import Libraries

In [ ]:
# Analisi dati e Visualizzazione
import numpy as np
import pandas as pd

# Scikit-learn: Preprocessing e Dimensionality Reduction
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder
)

# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline

# Scikit-learn: Modelli di Machine Learning Classico
from sklearn.ensemble import RandomForestClassifier

# Our packages
from support_modules.utils import *
from support_modules.plot import *

#Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.compose import make_column_selector
from sklearn import set_config
set_config(transform_output="pandas")

# Global Variables

In [ ]:
SEED = 42
FILENAME = "../data/train.csv"

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates rows and columns

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df.T.drop_duplicates(inplace=True).T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


## 2. Label extraction and train-test splitting

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

## 3. Pipeline

In [ ]:
from support_modules.preprocessing import *

In [ ]:
from support_modules.constants import *

In [ ]:
remainder_pipeline = Pipeline([
    ('impute_median', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('impute_unknown', SimpleImputer(strategy='constant', fill_value='Unknown'), categorical_to_unknown_cols),
        ('impute_0', SimpleImputer(strategy='constant', fill_value=0), fill_zero_cols),
        ('impute_10k', SimpleImputer(strategy='constant', fill_value=999), fill_big_cols),
        ('impute_mode_cat',SimpleImputer(strategy='most_frequent'), fill_to_mode_cat),
        ('impute_mode_num', SimpleImputer(strategy='most_frequent'), fill_to_mode_num),
    ],
    remainder=remainder_pipeline,
    verbose_feature_names_out=False
)


encoding = ColumnTransformer(
    transformers=[
        # Applica categorical_pipe a TUTTE le colonne di tipo object o category
        ('cat_step', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), make_column_selector(dtype_include=['object'])),
    ], 
    # Tutto ciò che è numerico e non è stato toccato sopra, viene scalato qui
    remainder="passthrough", 
    verbose_feature_names_out=False
)

In [ ]:

full_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns = loan_performance_data_leakage + settlement_data_leakage + hardship_data_leakage + other_leakage + loan_contract_interest_rate)),
    ('drop_non_significant', ColumnDropper(columns = other_non_significant)),
    ('drop_high_nan', HighNanDropper(threshold=0.9)),
    ('drop_joint_and_secondary', ColumnDropper(columns = joint_and_secondary_cols)),
    ('high_correlation', HighlyCorrelatedDropper(threshold=0.95)),
    ('feature_extraction', NumericExtractor(columns = number_from_string_cols)),
    ('fico_average', FeatureAverager(columns = average_cols)),
    ('date_diff', DateDifferenceTransformer(reference_col=date_diff_reference, target_cols=date_diff_target)),
    ('rounding_int', RoundToIntTransformer(columns = round_to_nearest_int)),

    ('preprocessor', preprocessor),
    ('encoding', encoding),
    
    ("clf", RandomForestClassifier(random_state=SEED, class_weight='balanced'))
])

param_grid_rf = {
    'clf__n_estimators': [250, 300, 350],
    'clf__criterion': ['gini', 'log_loss'],
    'clf__max_depth': [5, 10, 15]
}

grid = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid_rf,
    cv=3,
    scoring='balanced_accuracy',
    error_score='raise',
    n_jobs=1,
    verbose=3
)
grid.fit(X_train, y_train)

In [ ]:
# 1. Recuperiamo il miglior stimatore
full_best_pipeline = grid.best_estimator_

print("Migliori parametri:", grid.best_params_)
print(f"Miglior CV Score: {grid.best_score_:.4f}")

# 2. Dividiamo la Pipeline
# [:-1] prende tutto tranne l'ultimo step ('clf')
preprocessing_part = full_best_pipeline[:-1] 
# Accediamo al classificatore tramite il suo nome 'clf'
classifier_part = full_best_pipeline.named_steps['clf']

# 3. Salvataggio
try:
    with open("rf_preprocessor.save", "wb") as f:
        pickle.dump(preprocessing_part, f)

    with open("rf.save", "wb") as f:
        pickle.dump(classifier_part, f)
        
    print("\nFile salvati correttamente:")
    print("- 'rf_preprocessor.save' (Contiene tutti i drop e le trasformazioni)")
    print("- 'rf.save' (Contiene solo il RandomForestClassifier allenato)")
except Exception as e:
    print(f"Errore durante il salvataggio: {e}")

In [ ]:
evaluate_model(X_val, y_val, "rf")